# Legal Search Index on Supreme Court Judgments (1950–1951)

This notebook builds a lightweight search index over Supreme Court judgments from 1950 and 1951. It uses **PyMuPDF** to parse PDFs and **scikit‑learn's** `TfidfVectorizer` to create a simple term–frequency index. If the Kaggle dataset directory is unavailable (as in this testing environment), a sample dataset is used instead.

### 1. Environment Setup

In [1]:
# Install lightweight dependencies. In Kaggle these may already be present.
!pip install --quiet pymupdf tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 58.3 MB/s eta 0:00:00


### 2. Imports and Constants

In [2]:
import os
import re

import fitz  # PyMuPDF for PDF parsing
from tqdm import tqdm  # progress bar
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Base directory of the Supreme Court judgments dataset
DATA_DIR = "/kaggle/input/legal-dataset-sc-judgments-india-19502024/supreme_court_judgments"

# Years to index (restrict to early years for quick demo)
YEARS = ["2024", "2025"]

# Determine the effective directory. If DATA_DIR doesn't exist, fall back to a sample dataset.
if os.path.exists(DATA_DIR):
    effective_dir = DATA_DIR
else:
    effective_dir = "/home/oai/share/sample_dataset/supreme_court_judgments"
    print(f"Dataset directory {DATA_DIR} not found; using sample data at {effective_dir}")

# Separator for joining pages (two newline characters)
PAGE_SEP = chr(10) * 2





### 3. PDF Parsing

In [3]:
def extract_text_from_pdf(path: str) -> str:
    # Extract and clean text from a PDF judgment.
    doc = fitz.open(path)
    text_pages = []
    for page in doc:
        text = page.get_text("text")
        # Replace non-breaking spaces with regular spaces
        text = text.replace(" ", " ")
        # Strip each line and collapse multiple whitespace
        text = " ".join(line.strip() for line in text.splitlines())
        text = re.sub(r"\s+", " ", text)
        text_pages.append(text.strip())
    doc.close()
    # Join pages with the defined separator (two newline characters)
    return PAGE_SEP.join(text_pages)


In [4]:
# Demo: list files in the 1950 folder and extract text from the first PDF
sample_text = None
year_dir = os.path.join(effective_dir, "1950")
if os.path.exists(year_dir):
    pdf_files = [f for f in os.listdir(year_dir) if f.lower().endswith(".pdf")]
    if pdf_files:
        sample_path = os.path.join(year_dir, pdf_files[0])
        print(f"Reading sample PDF: {pdf_files[0]}")
        sample_text = extract_text_from_pdf(sample_path)
        print(sample_text[:1500])
    else:
        print("No PDF files found in 1950 directory.")
else:
    print("1950 directory not found.")


Reading sample PDF: Kapildeo_Singh_vs_The_King_on_24_January_1950_1.PDF
Kapildeo Singh vs The King on 24 January, 1950 Equivalent citations: [1950]SUPPSCR144 Bench: Meher Chand Mahajan, Saiyad Fazal Ali JUDGMENT Mahajan, J. 1. This is an appeal by special leave against an order of the High Court at Patna, affirming the conviction of the appellant by the Additional Sessions Judge, Arrah, under s. 147, of the Indian Penal Code. 2. The appellant was charged along with 13 others with having been a member of an unlawful assembly "with the common object of dispossessing one Chulhan Tewari (the complainant) and assaulting and murdering one Nasiba Ahir and others" and with having committed, in furtherance of that common object, offences under ss. 302, 326 and 147 read with s. 249 of the Indian Penal Code. The prosecution case was that the appellant led a party of 60 or 70 men armed with a gun and lathis to the scene of occurrence with a view to dispossess the complainant of the land bearing su

### 4. Chunking

In [5]:
def split_into_chunks(text: str, max_words: int = 200):
    # Split text into chunks of roughly `max_words` words, respecting paragraph boundaries.
    paras = [p.strip() for p in text.split(PAGE_SEP) if p.strip()]
    chunks = []
    current_words = []
    count = 0
    for p in paras:
        words = p.split()
        # If adding this paragraph would exceed max_words, start a new chunk
        if count + len(words) > max_words:
            if current_words:
                chunks.append(" ".join(current_words))
            current_words = words
            count = len(words)
        else:
            current_words.extend(words)
            count += len(words)
    if current_words:
        chunks.append(" ".join(current_words))
    return chunks


In [6]:
# Demo: chunk the sample text extracted earlier
if sample_text:
    demo_chunks = split_into_chunks(sample_text, max_words=200)
    print(f"Number of chunks: {len(demo_chunks)}")
    if demo_chunks:
        print(demo_chunks[0][:400])
else:
    print("No sample text available for chunking demo.")


Number of chunks: 5
Kapildeo Singh vs The King on 24 January, 1950 Equivalent citations: [1950]SUPPSCR144 Bench: Meher Chand Mahajan, Saiyad Fazal Ali JUDGMENT Mahajan, J. 1. This is an appeal by special leave against an order of the High Court at Patna, affirming the conviction of the appellant by the Additional Sessions Judge, Arrah, under s. 147, of the Indian Penal Code. 2. The appellant was charged along with 13


### 5. Build Corpus

In [7]:
all_chunks = []  # list to hold chunk texts
meta = []        # list to hold metadata dictionaries

total_pdfs = 0
for year in YEARS:
    year_path = os.path.join(effective_dir, year)
    if not os.path.exists(year_path):
        print(f"Warning: directory for year {year} not found")
        continue
    pdf_files = [f for f in os.listdir(year_path) if f.lower().endswith(".pdf")]
    print(f"Year {year}: {len(pdf_files)} PDFs found")
    for pdf_file in tqdm(pdf_files, desc=f"Processing {year}"):
        total_pdfs += 1
        pdf_path = os.path.join(year_path, pdf_file)
        try:
            text = extract_text_from_pdf(pdf_path)
        except Exception as e:
            print(f"Error reading {pdf_file}: {e}")
            continue
        if not text:
            continue
        chunks = split_into_chunks(text, max_words=200)
        for idx, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            meta.append({
                "year": year,
                "file": pdf_file,
                "chunk_id": idx
            })

print(f"Total PDFs processed: {total_pdfs}")
print(f"Total chunks collected: {len(all_chunks)}")
# Show first few metadata entries
print(meta[:2])


Year 2024: 400 PDFs found


Processing 2024: 100%|██████████| 400/400 [00:36<00:00, 10.83it/s]


Year 2025: 400 PDFs found


Processing 2025: 100%|██████████| 400/400 [00:24<00:00, 16.25it/s]

Total PDFs processed: 800
Total chunks collected: 14801
[{'year': '2024', 'file': 'Navas_Mulanavas_vs_State_Of_Kerala_on_18_March_2024_1.PDF', 'chunk_id': 0}, {'year': '2024', 'file': 'Navas_Mulanavas_vs_State_Of_Kerala_on_18_March_2024_1.PDF', 'chunk_id': 1}]


### 6. TF-IDF Index

In [8]:
if all_chunks:
    vectorizer = TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        stop_words="english"
    )
    tfidf_matrix = vectorizer.fit_transform(all_chunks)
    print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
else:
    vectorizer = None
    tfidf_matrix = None
    print("No chunks available; TF-IDF index not created.")


TF-IDF matrix shape: (14801, 50000)


### 7. Search API

In [9]:
def search(query: str, k: int = 5):
    # Return top-k most relevant chunks for the query using cosine similarity.
    if vectorizer is None or tfidf_matrix is None:
        return []
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    results = []
    for rank, idx in enumerate(top_idx):
        results.append({
            "rank": rank + 1,
            "score": float(sims[idx]),
            "text": all_chunks[idx],
            "meta": meta[idx],
        })
    return results

def pretty_print_results(results):
    for r in results:
        print("----")
        print(f"Rank: {r['rank']} | Score: {r['score']:.4f}")
        m = r['meta']
        print(f"Source: {m['file']} (year {m['year']}, chunk {m['chunk_id']})")
        # Print the first 500 characters of the text
        print(r['text'][:500])
        print()  # blank line for readability


### 8. Example Queries

In [10]:
# Run some example queries typical of early Supreme Court cases
queries = [
    "preventive detention and fundamental rights",
    "Article 19 freedom of speech",
    "habeas corpus under Article 32",
]
for q in queries:
    print()  # blank line before each query
    print("QUERY:", q)
    results = search(q, k=3)
    pretty_print_results(results)



QUERY: preventive detention and fundamental rights
----
Rank: 1 | Score: 0.4770
Source: Nenavath_Bujji_vs_The_State_Of_Telangana_on_21_March_2024_1.PDF (year 2024, chunk 14)
whether besides the person being a “GOONDA” his alleged activities are such which adversely affected the public order or are likely to affect the maintenance of public order. 24. The essential concept of preventive detention is that the detention of a person is not to punish him for something he has done but to prevent him from doing it. The basis of detention is the satisfaction of the executive about the likelihood of the detenu acting in a manner, similar to his past acts, which is likely to 

----
Rank: 2 | Score: 0.4648
Source: Joyi_Kitty_Joseph_vs_Union_Of_India_on_6_March_2025_1.PDF (year 2025, chunk 6)
detaining authority did not consider the efficacy of the conditions and enter any satisfaction, however subjective it is, as to the conditions not being sufficient to restrain the detenu from indulging in su

In [11]:
import re

def split_into_sentences(text: str):
    """Roughly split text into sentences using punctuation."""
    sentences = re.split(r'(?<=[.?!])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

# Demo: test sentence splitting on the first chunk
if all_chunks:
    sample_sentences = split_into_sentences(all_chunks[0])
    print(f"Number of sentences in first chunk: {len(sample_sentences)}")
    print(sample_sentences[:5])
else:
    print("No chunks available for sentence splitting demo.")


Number of sentences in first chunk: 24
['Navas @ Mulanavas vs State Of Kerala on 18 March, 2024 Author: B.', 'R.', 'Gavai Bench: B.', 'R.', 'Gavai 2024 INSC 215 REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO.']


In [12]:
from typing import List, Dict


def answer_query_extractive(query: str, k_chunks: int = 8, max_sentences: int = 7) -> Dict:
    """
    1. Run `search(query, k_chunks)` to get the most relevant chunks.
    2. Break those chunks into individual sentences.
    3. Score each sentence by TF-IDF cosine similarity to the query.
    4. Select the top `max_sentences` distinct sentences as an 'answer summary'.
    5. Also aggregate which cases (files) appeared in the retrieved chunks.

    Returns a dict with:
      - 'query'
      - 'summary_sentences': list[str]
      - 'cases': list[{'file': ..., 'year': ..., 'count': ...}]
      - 'raw_results': the original `search` results (for debugging)
    """
    # Perform search
    results = search(query, k=k_chunks)
    # If search returned nothing, return an empty structure
    if not results:
        return {
            'query': query,
            'summary_sentences': [],
            'cases': [],
            'raw_results': results
        }
    # Prepare query vector
    q_vec = vectorizer.transform([query])
    sentence_entries = []
    # Collect sentences and compute similarity scores
    for res in results:
        sentences = split_into_sentences(res['text'])
        for sent in sentences:
            # Compute vector for sentence
            sent_vec = vectorizer.transform([sent])
            # Cosine similarity between query and sentence
            sim = cosine_similarity(q_vec, sent_vec)[0][0]
            sentence_entries.append({
                'sentence': sent,
                'score': float(sim),
                'file': res['meta']['file'],
                'year': res['meta']['year'],
                'chunk_id': res['meta']['chunk_id']
            })
    # Sort sentences by similarity score in descending order
    sentence_entries.sort(key=lambda x: x['score'], reverse=True)
    # Deduplicate sentences and select top ones
    seen_sentences = set()
    summary_sentences = []
    for entry in sentence_entries:
        if entry['sentence'] not in seen_sentences:
            seen_sentences.add(entry['sentence'])
            summary_sentences.append(entry['sentence'])
            if len(summary_sentences) >= max_sentences:
                break
    # Aggregate case counts based on retrieved chunks
    case_counts = {}
    for res in results:
        key = (res['meta']['file'], res['meta']['year'])
        case_counts[key] = case_counts.get(key, 0) + 1
    cases = []
    for (file_name, year), count in case_counts.items():
        cases.append({'file': file_name, 'year': year, 'count': count})
    # Sort cases by frequency
    cases.sort(key=lambda x: x['count'], reverse=True)
    return {
        'query': query,
        'summary_sentences': summary_sentences,
        'cases': cases,
        'raw_results': results
    }


In [13]:
def print_answer(answer_dict):
    print('QUERY:', answer_dict['query'])
    print()  # blank line
    print('=== SUMMARY (extractive) ===')
    for i, sent in enumerate(answer_dict['summary_sentences'], 1):
        print(f"{i}. {sent}")
    print()  # blank line
    print('=== KEY CASES (from retrieved chunks) ===')
    for c in answer_dict['cases']:
        print(f"- {c['file']} (year {c['year']}), hits in top chunks: {c['count']}")


In [14]:
demo_queries = [
    'preventive detention and fundamental rights',
    'Article 19 freedom of speech and its relation with detention',
    'habeas corpus under Article 32'
]
for q in demo_queries:
    print('--------------------------------------------------')
    ans = answer_query_extractive(q, k_chunks=8, max_sentences=7)
    print_answer(ans)
    print()  # extra blank line


--------------------------------------------------
QUERY: preventive detention and fundamental rights

=== SUMMARY (extractive) ===
1. The power of preventive detention is qualitatively different from punitive detention.
2. An order of preventive detention, may be made before or during prosecution.
3. This proposition is valid both for punitive and preventive detention.
4. An order of preventive detention is also not a bar to prosecution.
5. The essential concept of preventive detention is that the detention of a person is not to punish him for something he has done but to prevent him from doing it.
6. These numbers evince a callous exercise of the exceptional power of preventive detention by the detaining authorities and the respondent-state.
7. The pendency of prosecution is no bar to an order of preventive detention.

=== KEY CASES (from retrieved chunks) ===
- Nenavath_Bujji_vs_The_State_Of_Telangana_on_21_March_2024_1.PDF (year 2024), hits in top chunks: 4
- Joyi_Kitty_Joseph_vs_U